# 05 Feature engineering with clinical validation (PB11)
Group 5 PROHI: Risk Factor Prediction of Chronic Kidney Disease

Teacher feedback (28 Sep): new features are allowed, but each needs clinical validation. This notebook builds three candidate features, checks how they relate to CKD, and compares model results with and without them.

Before running: in the cell below, replace `your-username` with your GitHub username. Then use Runtime > Run all.

In [2]:
GITHUB_USER = "mhoyor"

%cd /content
!rm -rf ckd-risk-app
!git clone https://github.com/mhoyor/ckd-risk-app.git
%cd /content/ckd-risk-app

/content
Cloning into 'ckd-risk-app'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 20 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 11.88 KiB | 1.48 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/ckd-risk-app


## Load the shared data (version 1)

In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from data_prep import load_data, FEATURES, LEAKAGE

d = load_data()

## Step 1: Build the three candidate features
Clinical rationale (details and sources in the PB11 Trello card):
- **comorbidity_count** = htn + dm + cad (0 to 3). Hypertension and diabetes are the leading causes of CKD; cardiovascular disease is strongly linked with CKD.
- **urine_sediment_count** = rbc + pc + pcc + ba (0 to 4). Urine sediment abnormalities are a marker of kidney damage in the KDIGO definition of CKD.
- **albuminuria_cat** = dipstick albumin grade grouped as 0 = A1 (negative), 1 = A2 (1+), 2 = A3 (2+ or more), approximating the KDIGO albuminuria categories.

Leakage rule: none of these use affected, stage or grf.

In [4]:
ENGINEERED = ["comorbidity_count", "urine_sediment_count", "albuminuria_cat"]

def add_candidates(df):
    df = df.copy()
    df["comorbidity_count"] = df["htn"] + df["dm"] + df["cad"]
    df["urine_sediment_count"] = df["rbc"] + df["pc"] + df["pcc"] + df["ba"]
    df["albuminuria_cat"] = df["al"].clip(upper=2)
    return df

all_v2 = add_candidates(d["all"])
train_v2 = add_candidates(d["train"])
all_v2[ENGINEERED].describe()

,comorbidity_count,urine_sediment_count,albuminuria_cat
count,200.000000,200.00000,200.00000
mean,0.850000,0.54000,0.73500
std,1.021133,0.91794,0.91045
min,0.000000,0.00000,0.00000
25%,0.000000,0.00000,0.00000
50%,0.000000,0.00000,0.00000
75%,2.000000,1.00000,2.00000
max,3.000000,3.00000,2.00000


## Step 2: How each feature relates to CKD


In [5]:
pd.crosstab(all_v2["class"], all_v2["comorbidity_count"])

comorbidity_count,0,1,2,3
class,,,,
0,72,0,0,0
1,35,30,49,14


In [6]:
pd.crosstab(all_v2["class"], all_v2["urine_sediment_count"])

urine_sediment_count,0,1,2,3
class,,,,
0,72,0,0,0
1,68,23,26,11


In [7]:
pd.crosstab(all_v2["class"], all_v2["albuminuria_cat"])

albuminuria_cat,0,1,2
class,,,
0,72,0,0
1,44,21,63


**Important finding for the report:** no non-CKD patient in this dataset has hypertension, diabetes, coronary artery disease, anaemia, albuminuria, glycosuria or any abnormal urine finding. The non-CKD group looks like a set of healthy controls rather than a typical clinic population. This partly explains the near-perfect model scores and limits how well the results would transfer to real screening, where many non-CKD patients have hypertension or diabetes.

## Step 3: Compare models with and without the engineered features
Stratified 5-fold cross-validation on the training set only (the test set stays untouched).

In [8]:
cv = StratifiedKFold(5, shuffle=True, random_state=42)
y = train_v2["class"]
models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
}
feature_sets = {
    "Original 25": FEATURES,
    "Original 25 + engineered": FEATURES + ENGINEERED,
}

rows = []
for model_name, model in models.items():
    for set_name, cols in feature_sets.items():
        scores = cross_validate(model, train_v2[cols], y, cv=cv,
                                scoring=["recall", "roc_auc", "f1", "accuracy"])
        rows.append({
            "Model": model_name,
            "Features": set_name,
            "Recall (sensitivity)": round(scores["test_recall"].mean(), 3),
            "ROC-AUC": round(scores["test_roc_auc"].mean(), 3),
            "F1": round(scores["test_f1"].mean(), 3),
            "Accuracy": round(scores["test_accuracy"].mean(), 3),
        })

results = pd.DataFrame(rows)
results

,Model,Features,Recall (sensitivity),ROC-AUC,F1,Accuracy
0,Logistic regression,Original 25,0.98,0.997,0.985,0.981
1,Logistic regression,Original 25 + engineered,1.00,0.998,0.995,0.994
2,Decision tree (depth 3),Original 25,0.98,0.965,0.976,0.969
3,Decision tree (depth 3),Original 25 + engineered,0.96,0.954,0.965,0.956


## Step 4: Decision (write it here and in the PB11 Trello card)
Interpretation: the engineered features give at most a very small change, because the original features already separate the groups almost perfectly. The value of the features is mainly **interpretability**: rules such as "2 or more comorbidities" or "albuminuria category A2 or higher" are easier for a care coordinator to use than individual coded variables.